# 06 · K-nearest neighbors (sampled feasibility baseline)

This notebook adds KNN to the PaySim model comparisons. Exact KNN prediction over the full multi-million-row training and validation partitions is computationally expensive, so this experiment uses deterministic stratified samples from the existing training and validation partitions. **Its scores are not directly comparable to the full-validation scores in notebooks 03–05.**

The feature set, dataset fingerprint and 70/15/15 split match the earlier notebooks. Scaling and encoding are fitted only on the sampled training rows. The reserved test partition is not used. KNN settings are initial choices, not a tuned optimum.

PaySim is synthetic, and the random transaction split is not chronological or account-disjoint. Earlier exploratory analysis used the full dataset.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
import sklearn, joblib
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             precision_recall_curve, precision_score, recall_score, roc_auc_score)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

# Resolve src from the project root, regardless of the kernel working directory.
_cwd = Path.cwd().resolve()
ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "data" / "README.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the project folder.")
sys.path.insert(0, str(ROOT / "src"))
from paydev import (SEED, FEATURES, NUMERIC, CATEGORICAL, SPLIT, find_root,
                    data_path, load_dataset, stratified_splits)
ROOT = find_root(_cwd)
print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}")


## 1. Load data and reproduce the split

The source CSV is fingerprint-verified. The original training and validation row IDs are recreated before sampling; test rows remain reserved.


In [ ]:
df, actual_sha256 = load_dataset(ROOT)
y = df["isFraud"].to_numpy(copy=True)
train_ids, val_ids, test_ids = stratified_splits(y, seed=SEED)
print("Dataset SHA-256:", actual_sha256)
print(f"Original partitions: train={len(train_ids):,}, validation={len(val_ids):,}, test reserved={len(test_ids):,}")


## 2. Make deterministic stratified samples

Use up to 50,000 training rows and 100,000 validation rows. Stratification preserves the rare class approximately, while sampling keeps exact neighbor search manageable. The validation sample is drawn only from the original validation partition.


In [ ]:
TRAIN_SAMPLE_SIZE = 50_000
VALIDATION_SAMPLE_SIZE = 100_000

def stratified_sample_ids(ids, labels, sample_size, seed):
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in (0, 1):
        class_ids = ids[labels[ids] == cls]
        n = min(len(class_ids), int(round(sample_size * len(class_ids) / len(ids))))
        chosen.append(rng.choice(class_ids, size=n, replace=False))
    result = np.concatenate(chosen)
    rng.shuffle(result)
    return result

train_sample_ids = stratified_sample_ids(train_ids, y, TRAIN_SAMPLE_SIZE, SEED)
val_sample_ids = stratified_sample_ids(val_ids, y, VALIDATION_SAMPLE_SIZE, SEED + 1)
print(f"Sampled training rows: {len(train_sample_ids):,}; fraud={int(y[train_sample_ids].sum()):,} ({y[train_sample_ids].mean():.4%})")
print(f"Sampled validation rows: {len(val_sample_ids):,}; fraud={int(y[val_sample_ids].sum()):,} ({y[val_sample_ids].mean():.4%})")


## 3. Fit scaled KNN

Numeric inputs are standardized and transaction type is one-hot encoded using sampled training rows only. `kd_tree` is used for neighbor lookup; it may still be slow as feature dimensions grow. The choice of 15 neighbors and distance weighting is a starting point, not a tuned result.


In [ ]:
preprocess = ColumnTransformer([
    ("numeric", StandardScaler(), NUMERIC),
    ("type", OneHotEncoder(handle_unknown="ignore", dtype=np.float32, sparse_output=False), CATEGORICAL),
], remainder="drop", sparse_threshold=0)
model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", KNeighborsClassifier(n_neighbors=15, weights="distance", algorithm="kd_tree", n_jobs=1)),
])
X_train = df.iloc[train_sample_ids][FEATURES]
y_train = y[train_sample_ids]
X_val = df.iloc[val_sample_ids][FEATURES]
y_val = y[val_sample_ids]
start = time.perf_counter()
with threadpool_limits(limits=2):
    model.fit(X_train, y_train)
fit_seconds = time.perf_counter() - start
print(f"Fit time: {fit_seconds:.1f} seconds")


## 4. Sampled validation metrics

Average precision (AP) is the step-wise precision–recall summary and differs from trapezoidal PR-AUC. The F1 threshold is selected on this sampled validation set, so its F1 is a selection result and can be optimistic. Neither threshold nor metrics are test results.


In [ ]:
start = time.perf_counter()
with threadpool_limits(limits=2):
    val_scores = model.predict_proba(X_val)[:, 1]
predict_seconds = time.perf_counter() - start
precision, recall, thresholds = precision_recall_curve(y_val, val_scores)
f1_values = np.divide(2 * precision[:-1] * recall[:-1], precision[:-1] + recall[:-1],
                      out=np.zeros_like(thresholds), where=(precision[:-1] + recall[:-1]) != 0)
best_idx = int(np.argmax(f1_values))
selected_threshold = float(thresholds[best_idx])

def measure(threshold):
    pred = val_scores >= threshold
    tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
    return {"threshold": threshold, "precision": precision_score(y_val, pred, zero_division=0),
            "recall": recall_score(y_val, pred, zero_division=0), "F1": f1_score(y_val, pred, zero_division=0),
            "false_positives": int(fp), "false_negatives": int(fn), "alerts": int(fp + tp),
            "alert_rate_percent": 100 * (fp + tp) / len(y_val)}

ap = float(average_precision_score(y_val, val_scores))
roc_auc = float(roc_auc_score(y_val, val_scores))
results = pd.DataFrame([
    {"operating_point": label, "average_precision": ap, "ROC_AUC": roc_auc, **measure(threshold)}
    for label, threshold in [("Default 0.50", 0.5), ("F1 selected on sampled validation", selected_threshold)]
]).set_index("operating_point")
display(results.round(4))
print(f"Sampled validation prevalence: {y_val.mean():.4%}; rows={len(y_val):,}; prediction time={predict_seconds:.1f}s")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.step(recall, precision, where="post", color="#2563a6", lw=2, label=f"KNN (AP={ap:.4f})")
ax.axhline(y_val.mean(), color="#a54444", linestyle="--", label=f"Sample prevalence ({y_val.mean():.3%})")
ax.scatter(recall[best_idx], precision[best_idx], color="#d95b43", zorder=3, label="Sampled validation F1 threshold")
ax.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02), title="KNN precision–recall · sampled validation")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 5. Save the KNN checkpoint

The checkpoint records the sampling procedure and settings so its scope is clear. It is generated locally and ignored by Git by default. Only load joblib files from trusted sources.


In [ ]:
checkpoint = {
    "pipeline": model, "model_name": "K-nearest neighbors",
    "threshold": selected_threshold, "threshold_selection": "max F1 on sampled validation",
    "features": FEATURES, "seed": SEED, "split": SPLIT, "dataset_sha256": actual_sha256,
    "training_sample_size": len(train_sample_ids), "validation_sample_size": len(val_sample_ids),
    "training_sampling": "stratified sample from training partition",
    "validation_sampling": "stratified sample from validation partition",
    "n_neighbors": 15, "weights": "distance", "algorithm": "kd_tree",
    "validation_average_precision": ap, "validation_roc_auc": roc_auc,
    "python_version": sys.version.split()[0], "numpy_version": np.__version__,
    "pandas_version": pd.__version__, "sklearn_version": sklearn.__version__,
}
checkpoint_path = ROOT / "models" / "knn_baseline.joblib"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(checkpoint, checkpoint_path, compress=3)
print(f"Saved {checkpoint_path.relative_to(ROOT).as_posix()} ({checkpoint_path.stat().st_size:,} bytes)")
print("The reserved test partition was not scored.")


## Next steps

Consider a smaller or approximate-neighbor method if KNN is too slow, and tune the sample size and K using training/validation data. For a fair comparison with the other models, assess KNN on the full validation partition only if the computational cost is acceptable; reserve the test partition until model selection is complete.
